In [1]:
from __future__ import annotations

import copy
import csv
import json
import os
import random
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Callable

from pathlib import Path
import sys
import os


try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()

for package_dir in (ROOT / ".torch_python", ROOT / ".project_python"):
    sys.path.insert(0, str(package_dir))

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib"))
for package_dir in (ROOT / ".torch_python", ROOT / ".project_python"):
    sys.path.insert(0, str(package_dir))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import torch
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from torchvision.models import SqueezeNet1_1_Weights

In [2]:
SEED = 42
IMAGE_SIZE = 96
BATCH_SIZE = 64
HEAD_EPOCHS = 4
FINE_TUNE_EPOCHS = 4
CLASS_IDS = (0, 1, 2)  # MNIST: zero, one, two
CLASS_NAMES = ("zero", "one", "two")
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

DATA_DIR = ROOT / "mnist_data"
OUTPUT_DIR = ROOT / "transfer_learning_mnist_outputs"
FIG_DIR = OUTPUT_DIR / "figures"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
MODEL_PATH = OUTPUT_DIR / "mnist012_squeezenet_transfer.pt"


@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = SEED
    image_size: int = IMAGE_SIZE
    batch_size: int = BATCH_SIZE
    head_epochs: int = HEAD_EPOCHS
    fine_tune_epochs: int = FINE_TUNE_EPOCHS
    train_per_class: int = 300
    validation_per_class: int = 75
    test_per_class: int = 150
    classes: tuple[str, ...] = CLASS_NAMES
    backbone: str = "SqueezeNet 1.1 pretrained on ImageNet"


CONFIG = ExperimentConfig()


class ReliableMNIST(datasets.MNIST):
    """MNIST dataset pinned to the available official OSSCI mirror."""

    mirrors = ["https://ossci-datasets.s3.amazonaws.com/mnist/"]


class RemappedSubset(Dataset):
    """Subset of MNIST with selected labels remapped to [0, number_of_classes)."""

    def __init__(self, base_dataset: Dataset, indices: list[int], label_map: dict[int, int], transform: Callable):
        self.base_dataset = base_dataset
        self.indices = indices
        self.label_map = label_map
        self.transform = transform

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, item: int):
        image, original_label = self.base_dataset[self.indices[item]]
        return self.transform(image), self.label_map[int(original_label)]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(False)


def make_transforms():
    """Return augmentation-rich training and deterministic evaluation transforms."""
    train_transform = transforms.Compose(
        [
            transforms.RandomAffine(degrees=12, translate=(0.10, 0.10), scale=(0.90, 1.10)),
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.78, 1.0), ratio=(0.95, 1.05)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )
    eval_transform = transforms.Compose(
        [
            transforms.Resize(int(IMAGE_SIZE * 1.15)),
            transforms.CenterCrop(IMAGE_SIZE),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )
    return train_transform, eval_transform


def make_balanced_indices(targets: list[int], per_class: int, offset: int, seed: int) -> list[int]:
    """Pick deterministic, balanced indices for selected MNIST labels."""
    rng = random.Random(seed)
    indices: list[int] = []
    for class_id in CLASS_IDS:
        available = [idx for idx, target in enumerate(targets) if target == class_id]
        rng.shuffle(available)
        indices.extend(available[offset : offset + per_class])
    rng.shuffle(indices)
    return indices


def build_datasets():
    DATA_DIR.mkdir(exist_ok=True)
    base_train = ReliableMNIST(root=str(DATA_DIR), train=True, download=True)
    base_test = ReliableMNIST(root=str(DATA_DIR), train=False, download=True)
    train_transform, eval_transform = make_transforms()
    label_map = {original: mapped for mapped, original in enumerate(CLASS_IDS)}

    train_indices = make_balanced_indices(base_train.targets, CONFIG.train_per_class, offset=0, seed=SEED)
    val_indices = make_balanced_indices(
        base_train.targets, CONFIG.validation_per_class, offset=CONFIG.train_per_class, seed=SEED
    )
    test_indices = make_balanced_indices(base_test.targets, CONFIG.test_per_class, offset=0, seed=SEED)

    return (
        base_train,
        RemappedSubset(base_train, train_indices, label_map, train_transform),
        RemappedSubset(base_train, val_indices, label_map, eval_transform),
        RemappedSubset(base_test, test_indices, label_map, eval_transform),
    )


def unnormalize(image_tensor: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    image = (image_tensor.cpu() * std + mean).clamp(0, 1)
    return image.permute(1, 2, 0).numpy()


def plot_augmentation_preview(base_train, train_dataset) -> None:
    """Show original MNIST images beside random augmented training views."""
    fig, axes = plt.subplots(3, 4, figsize=(10.5, 7.5))
    for row, item in enumerate((0, 1, 2)):
        original_index = train_dataset.indices[item]
        original_image, original_label = base_train[original_index]
        axes[row, 0].imshow(original_image)
        axes[row, 0].set_title(f"Original: {CLASS_NAMES[int(original_label)]}", fontsize=9)
        axes[row, 0].axis("off")
        for col in range(1, 4):
            augmented, mapped_label = train_dataset[item]
            axes[row, col].imshow(unnormalize(augmented))
            axes[row, col].set_title(f"Augmented view {col}", fontsize=9)
            axes[row, col].axis("off")
            assert CLASS_NAMES[mapped_label] == CLASS_NAMES[int(original_label)]
    fig.suptitle("Training augmentation: affine shift, crop, ImageNet normalization", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "01_augmentation_preview.png", bbox_inches="tight", dpi=220)
    plt.close(fig)


def build_model(num_classes: int) -> nn.Module:
    """Load ImageNet weights, freeze the backbone, and replace the classification head."""
    model = models.squeezenet1_1(weights=SqueezeNet1_1_Weights.DEFAULT)
    for parameter in model.features.parameters():
        parameter.requires_grad = False
    model.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=1)
    model.num_classes = num_classes
    return model


def set_fine_tuning(model: nn.Module) -> None:
    """Unfreeze the final convolutional stage for a cautious second training phase."""
    for parameter in model.features[10:].parameters():
        parameter.requires_grad = True


def run_epoch(model, loader, criterion, optimizer, device, training: bool):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_labels, all_predictions = [], []
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            all_labels.extend(labels.detach().cpu().numpy())
            all_predictions.extend(logits.argmax(dim=1).detach().cpu().numpy())
    return total_loss / len(loader.dataset), accuracy_score(all_labels, all_predictions)


def plot_training_curves(history: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="Training")
    axes[0].plot(history["epoch"], history["val_loss"], marker="o", label="Validation")
    axes[0].set(title="Loss by epoch", xlabel="Epoch", ylabel="Cross-entropy loss")
    axes[0].legend()
    axes[1].plot(history["epoch"], history["train_accuracy"], marker="o", label="Training")
    axes[1].plot(history["epoch"], history["val_accuracy"], marker="o", label="Validation")
    axes[1].set(title="Accuracy by epoch", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1.02))
    axes[1].legend()
    for ax in axes:
        ax.axvline(HEAD_EPOCHS + 0.5, color="gray", ls="--", lw=1)
        ax.text(HEAD_EPOCHS + 0.58, ax.get_ylim()[0] + (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.07, "fine-tuning", color="gray", fontsize=9)
    fig.suptitle("Transfer-learning training history", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "02_training_curves.png", bbox_inches="tight", dpi=220)
    plt.close(fig)


def evaluate_model(model, loader, device):
    model.eval()
    all_labels, all_predictions, all_probabilities, all_images = [], [], [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            probabilities = torch.softmax(logits, dim=1).cpu().numpy()
            all_labels.extend(labels.numpy())
            all_predictions.extend(probabilities.argmax(axis=1))
            all_probabilities.extend(probabilities)
            all_images.extend(images.numpy())
    return (
        np.asarray(all_labels),
        np.asarray(all_predictions),
        np.asarray(all_probabilities),
        np.asarray(all_images),
    )


def plot_evaluation(y_true, y_pred, probabilities, images) -> tuple[dict, str]:
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "roc_auc_ovr_macro": float(roc_auc_score(y_true, probabilities, multi_class="ovr", average="macro")),
    }
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0)

    fig, ax = plt.subplots(figsize=(7.2, 6))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=CLASS_NAMES, normalize="true", values_format=".2f", cmap="Blues", colorbar=False, ax=ax)
    ax.set_title("Held-out test set: normalized confusion matrix", pad=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "03_test_confusion_matrix.png", bbox_inches="tight", dpi=220)
    plt.close(fig)

    y_binary = label_binarize(y_true, classes=np.arange(len(CLASS_NAMES)))
    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    palette = sns.color_palette("tab10", n_colors=len(CLASS_NAMES))
    for idx, (name, color) in enumerate(zip(CLASS_NAMES, palette)):
        fpr, tpr, _ = roc_curve(y_binary[:, idx], probabilities[:, idx])
        ax.plot(fpr, tpr, color=color, lw=2, label=f"{name.title()} (AUC={auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax.set(title="Held-out test ROC curves (one-vs-rest)", xlabel="False positive rate", ylabel="True positive rate", xlim=(-0.01, 1), ylim=(0, 1.02))
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "04_test_roc_curves.png", bbox_inches="tight", dpi=220)
    plt.close(fig)

    confidence = probabilities.max(axis=1)
    chosen = np.argsort(-confidence)[:12]
    fig, axes = plt.subplots(3, 4, figsize=(10.5, 8))
    for ax, idx in zip(axes.ravel(), chosen):
        ax.imshow(unnormalize(torch.from_numpy(images[idx])))
        correct = y_true[idx] == y_pred[idx]
        colour = "#237A3B" if correct else "#B22222"
        ax.set_title(
            f"true: {CLASS_NAMES[y_true[idx]]}\npred: {CLASS_NAMES[y_pred[idx]]} ({confidence[idx]:.0%})",
            fontsize=9,
            color=colour,
        )
        ax.axis("off")
    fig.suptitle("Most confident held-out predictions", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "05_test_predictions.png", bbox_inches="tight", dpi=220)
    plt.close(fig)
    return metrics, report


def save_checkpoint(model, device, history, metrics) -> None:
    checkpoint = {
        "architecture": "squeezenet1_1",
        "pretrained_backbone": "ImageNet SqueezeNet 1.1",
        "class_names": list(CLASS_NAMES),
        "image_size": IMAGE_SIZE,
        "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
        "state_dict": {key: value.cpu() for key, value in model.state_dict().items()},
        "best_validation_accuracy": float(history["val_accuracy"].max()),
        "test_metrics": metrics,
        "device_used_for_training": str(device),
        "seed": SEED,
    }
    torch.save(checkpoint, MODEL_PATH)


def write_report(history: pd.DataFrame, metrics: dict, classification_text: str, device: torch.device) -> None:
    metrics_json = {
        "config": asdict(CONFIG),
        "device": str(device),
        "best_epoch": int(history.loc[history["val_accuracy"].idxmax(), "epoch"]),
        "best_validation_accuracy": float(history["val_accuracy"].max()),
        "test_metrics": metrics,
        "torch_version": torch.__version__,
    }
    (ARTIFACT_DIR / "test_metrics.json").write_text(json.dumps(metrics_json, indent=2), encoding="utf-8")
    (ARTIFACT_DIR / "test_classification_report.txt").write_text(classification_text, encoding="utf-8")
    history.to_csv(ARTIFACT_DIR / "training_history.csv", index=False)

    report = f"""# Transfer Learning: MNIST Three-Class Image Classifier

| Metric | Held-out test result |
|---|---:|
| Accuracy | {metrics['accuracy']:.4f} |
| Precision (macro) | {metrics['precision_macro']:.4f} |
| Recall (macro) | {metrics['recall_macro']:.4f} |
| F1 (macro) | {metrics['f1_macro']:.4f} |
| ROC-AUC (one-vs-rest macro) | {metrics['roc_auc_ovr_macro']:.4f} |



- **Data:** a deterministic, balanced subset of MNIST: 900 training, 225 validation, and 450 test images (three classes).
- **Augmentation:** random affine shift/rotation and random resized crop on training images; deterministic resize/centre crop at evaluation.
- **Transfer learning:** ImageNet weights initialize SqueezeNet 1.1. The feature extractor is frozen during epochs 1–{HEAD_EPOCHS}; the final feature stage is unfrozen for epochs {HEAD_EPOCHS + 1}–{HEAD_EPOCHS + FINE_TUNE_EPOCHS}.
- **Model selection:** the checkpoint with the best validation accuracy (epoch {int(history.loc[history['val_accuracy'].idxmax(), 'epoch'])}; {history['val_accuracy'].max():.4f}) is saved and evaluated once on the test set.
- **Reproducibility:** fixed seed {SEED}; device used: `{device}`; PyTorch {torch.__version__}.

## Evidence

See `figures/01_augmentation_preview.png` for the actual data augmentation, `figures/02_training_curves.png` for optimization behaviour, and the confusion matrix / ROC curves for class-level test performance. The checkpoint stores class labels, ImageNet normalization values, image size, architecture, and weights so `inference.py` can load it without retraining.

## Interpretation and limitations

The model is intentionally small and fast, and the test set is a controlled MNIST subset. These numbers should not be interpreted as performance on arbitrary handwritten text or photographs. A next iteration would expand to all ten MNIST classes, tune the fine-tuning depth, and validate on a distinct digit dataset.
"""
    (OUTPUT_DIR / "Transfer_Learning_Report.md").write_text(report, encoding="utf-8")


def main() -> None:
    set_seed(SEED)
    torch.set_num_threads(min(4, os.cpu_count() or 1))
    OUTPUT_DIR.mkdir(exist_ok=True)
    FIG_DIR.mkdir(exist_ok=True)
    ARTIFACT_DIR.mkdir(exist_ok=True)
    sns.set_theme(style="whitegrid", context="notebook")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}; PyTorch {torch.__version__}")
    print("Downloading/loading MNIST and preparing balanced subsets...")
    base_train, train_dataset, val_dataset, test_dataset = build_datasets()
    plot_augmentation_preview(base_train, train_dataset)
    print(f"Train / validation / test images: {len(train_dataset)} / {len(val_dataset)} / {len(test_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    model = build_model(num_classes=len(CLASS_NAMES)).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    history_rows = []
    best_accuracy = -float("inf")
    best_state = None
    epoch_number = 0

    phases = [("head only", HEAD_EPOCHS, 2e-3), ("fine tune final stage", FINE_TUNE_EPOCHS, 2e-4)]
    for phase_name, phase_epochs, learning_rate in phases:
        if phase_name.startswith("fine"):
            set_fine_tuning(model)
        optimizer = AdamW((parameter for parameter in model.parameters() if parameter.requires_grad), lr=learning_rate, weight_decay=1e-4)
        for _ in range(phase_epochs):
            epoch_number += 1
            train_loss, train_accuracy = run_epoch(model, train_loader, criterion, optimizer, device, training=True)
            val_loss, val_accuracy = run_epoch(model, val_loader, criterion, optimizer, device, training=False)
            history_rows.append(
                {
                    "epoch": epoch_number,
                    "phase": phase_name,
                    "learning_rate": learning_rate,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "train_accuracy": train_accuracy,
                    "val_accuracy": val_accuracy,
                }
            )
            marker = "  <-- best validation checkpoint" if val_accuracy > best_accuracy else ""
            print(f"Epoch {epoch_number:02d} | {phase_name:21s} | train acc {train_accuracy:.4f} | val acc {val_accuracy:.4f}{marker}")
            if val_accuracy > best_accuracy:
                best_accuracy = val_accuracy
                best_state = copy.deepcopy(model.state_dict())

    assert best_state is not None
    model.load_state_dict(best_state)
    history = pd.DataFrame(history_rows)
    plot_training_curves(history)
    y_true, y_pred, probabilities, images = evaluate_model(model, test_loader, device)
    metrics, classification_text = plot_evaluation(y_true, y_pred, probabilities, images)
    save_checkpoint(model, device, history, metrics)
    write_report(history, metrics, classification_text, device)

    print("\nTraining complete.")
    print(f"Best validation accuracy: {best_accuracy:.4f}")
    print("Held-out test metrics:")
    for name, value in metrics.items():
        print(f"  {name}: {value:.4f}")
    print(f"Saved checkpoint: {MODEL_PATH}")
    print(f"Saved report: {OUTPUT_DIR / 'Transfer_Learning_Report.md'}")


if __name__ == "__main__":
    main()

Using device: cpu; PyTorch 2.13.0+cpu
Downloading/loading MNIST and preparing balanced subsets...
Train / validation / test images: 900 / 225 / 450
Epoch 01 | head only             | train acc 0.8022 | val acc 0.9600  <-- best validation checkpoint
Epoch 02 | head only             | train acc 0.9500 | val acc 0.9867  <-- best validation checkpoint
Epoch 03 | head only             | train acc 0.9700 | val acc 0.9867
Epoch 04 | head only             | train acc 0.9678 | val acc 0.9867
Epoch 05 | fine tune final stage | train acc 0.9689 | val acc 0.9822
Epoch 06 | fine tune final stage | train acc 0.9778 | val acc 0.9867
Epoch 07 | fine tune final stage | train acc 0.9889 | val acc 0.9867
Epoch 08 | fine tune final stage | train acc 0.9911 | val acc 0.9911  <-- best validation checkpoint

Training complete.
Best validation accuracy: 0.9911
Held-out test metrics:
  accuracy: 0.9911
  precision_macro: 0.9912
  recall_macro: 0.9911
  f1_macro: 0.9911
  roc_auc_ovr_macro: 0.9991
Saved checkpo

In [ ]:
from pathlib import Path

import torch
from torch import nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

# =====================================================
# PATHS
# =====================================================

CHECKPOINT_PATH = Path(
    r"C:\Users\hp\internspark\transfer_learning_mnist_outputs\mnist012_squeezenet_transfer.pt"
)

IMAGE_PATH = Path(r"C:\Users\hp\Pictures\7.png")     # Change if your image is elsewhere

# =====================================================
# CHECK FILES
# =====================================================

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Model not found:\n{CHECKPOINT_PATH}"
    )

if not IMAGE_PATH.exists():
    raise FileNotFoundError(
        f"Image not found:\n{IMAGE_PATH}"
    )

# =====================================================
# LOAD CHECKPOINT
# =====================================================

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

class_names = checkpoint["class_names"]

# =====================================================
# CREATE MODEL
# =====================================================

model = models.squeezenet1_1(weights=None)

model.classifier[1] = nn.Conv2d(
    512,
    len(class_names),
    kernel_size=1
)

model.num_classes = len(class_names)

model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# IMAGE TRANSFORM
# =====================================================

transform = transforms.Compose([
    transforms.Resize(int(checkpoint["image_size"] * 1.15)),
    transforms.CenterCrop(checkpoint["image_size"]),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        checkpoint["normalization"]["mean"],
        checkpoint["normalization"]["std"]
    ),
])

# =====================================================
# LOAD IMAGE
# =====================================================

image = Image.open(IMAGE_PATH).convert("L")

input_tensor = transform(image).unsqueeze(0)

# =====================================================
# PREDICT
# =====================================================

with torch.no_grad():
    output = model(input_tensor)
    probabilities = torch.softmax(output, dim=1)[0]

scores, indices = torch.topk(
    probabilities,
    len(class_names)
)

# =====================================================
# SHOW IMAGE
# =====================================================

plt.figure(figsize=(4,4))
plt.imshow(image, cmap="gray")
plt.title("Input Image")
plt.axis("off")
plt.show()

# =====================================================
# RESULTS
# =====================================================

print("=" * 45)
print("Prediction Results")
print("=" * 45)

for rank, (score, idx) in enumerate(
    zip(scores.tolist(), indices.tolist()), 1
):
    print(f"{rank}. {class_names[idx]} : {score:.2%}")

print("\nPredicted Digit:", class_names[indices[0].item()])